# ⚙️ 03 - Özellik Mühendisliği (Segment-Level Feature Engineering)
## Uydu Telemetri Anomali Tespiti

**Amaç:** Zaman serisindeki saniyelik gürültüleri ortadan kaldırmak için anomali tespitini **Olay (Segment) Bazlı** hale getirmek. ESA'nın orijinal özellik matrisi (`dataset.csv`) ile kendi ürettiğimiz Sinyal İşleme özelliklerini (RMS, Peak-to-Peak, ZCR) birleştireceğiz.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import json
import sys
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

sys.path.insert(0, '..')
from src.feature_engineer import ReactionWheelFeatureEngineer

print('✅ Kütüphaneler ve Feature Engineer yüklendi.')


---
## 📥 Veri Yükleme (Raw Segments & Pre-Extracted Features)


In [ ]:
# Orijinal zaman serisi verisini yükle (Özel özellikler çıkarmak için)
df_segments = pd.read_csv('../data/raw/segments.csv')

# ESA tarafından hazırlanan segment-bazlı istatistiksel özellikleri yükle
df_dataset = pd.read_csv('../data/raw/dataset.csv')

print(f'📊 Zaman Serisi (Segments) Boyutu: {df_segments.shape}')
print(f'📊 Hedef Olay (Dataset) Boyutu: {df_dataset.shape}')


---
## 🚀 Segment Bazlı Özel Özellik Çıkarımı (Custom Feature Extraction)
Her bir olay (segment) için Sinyal İşleme metriklerini hesaplıyoruz.


In [ ]:
engineer = ReactionWheelFeatureEngineer()

# df_segments üzerinden her segment için RMS, P2P, Crest Factor, ZCR hesapla
df_custom_features = engineer.extract_segment_features(df_segments)

print('\n=== Üretilen Özel Özelliklerin İlk 5 Satırı ===')
display(df_custom_features.head())


---
## 🔄 Özellik Birleştirme (Data Merging)
ESA'nın sağladığı `mean`, `var`, `skew` gibi istatistiksel özelliklerle, bizim ürettiğimiz `custom_rms`, `custom_zcr` gibi Sinyal özelliklerini birleştiriyoruz.


In [ ]:
# Dataset ile Custom özelliklerimizi 'segment' sütunu üzerinden birleştiriyoruz
# İki tabloda da 'anomaly' ve 'channel' var, çakışmayı önlemek için drop ediyoruz
df_custom_features = df_custom_features.drop(columns=['anomaly', 'channel'])

df_final = pd.merge(df_dataset, df_custom_features, on='segment', how='inner')

# Kanal (Sensör) adlarını modellerin anlayabilmesi için sayısal ID'lere çevirelim
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_final['channel_id'] = le.fit_transform(df_final['channel'])

print(f"✅ Birleştirilmiş Zengin Veri Seti Boyutu: {df_final.shape}")
display(df_final.head(3))


---
## 💾 Özellik Matrisini ve Kataloğu Kaydetme


In [ ]:
# Tüm segmentlerin features matrisini kaydet.
df_final.to_parquet('../data/features/segment_features.parquet')

catalog = {
    "total_segments": len(df_final),
    "features_list": list(df_final.columns),
    "method": "Segment-Level Aggregation & Merging"
}
with open('../data/features/feature_catalog.json', 'w', encoding='utf-8') as f:
    json.dump(catalog, f, indent=4, ensure_ascii=False)

print('✅ Olay Bazlı (Segment-Level) Özellik Matrisi başarıyla kaydedildi.')


### HTML Rapor Export


In [ ]:
%pip install jupyter nbconvert -q
!jupyter nbconvert --to html 03_feature_engineering.ipynb --output ../reports/03_feature_engineering_rapor.html
print("HTML Raporu kaydedildi.")
